In [88]:
# Import packages and load dataset
import pandas as pd
import numpy as np
sales_df = pd.read_csv('retail_store_sales.csv')

Let's first get a feel for what our dataset looks like, and the amount of rows/columns.

In [89]:
sales_df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [90]:
# Print number of rows/columns of df
sales_df.shape

(12575, 11)

Now, let's look for duplicate rows and null values in the dataset.

In [91]:
# Count duplicate rows, amount of nulls in each column
duplicate_rows = sales_df.duplicated().sum()
print("Duplicates rows:", duplicate_rows)

null_counts = sales_df.isnull().sum()
print(null_counts)

Duplicates rows: 0
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


Now, let's dive deeper into each column to check data types and possible incorrect values. It should be noted that null values are classified as the "float" data type.

In [92]:
# Check data type percentages for each column
dtype_percentages = pd.DataFrame({
    col: sales_df[col].apply(lambda x: type(x).__name__).value_counts(normalize=True) * 100
    for col in sales_df.columns
}).T.fillna(0)

dtype_percentages

,bool,float,str
Transaction ID,0.00000,0.000000,100.000000
Customer ID,0.00000,0.000000,100.000000
Category,0.00000,0.000000,100.000000
Item,0.00000,9.646123,90.353877
Price Per Unit,0.00000,100.000000,0.000000
Quantity,0.00000,100.000000,0.000000
Total Spent,0.00000,100.000000,0.000000
Payment Method,0.00000,0.000000,100.000000
Location,0.00000,0.000000,100.000000
Transaction Date,0.00000,0.000000,100.000000


In [93]:
# Check for typos in Category, Payment Method, and Location columns
category_unique = sales_df['Category'].unique()
payment_unique = sales_df['Payment Method'].unique()
location_unique = sales_df['Location'].unique()

print("Unique values in Category:", category_unique)
print("Unique values in Payment Method:", payment_unique)
print("Unique values in Location:", location_unique)

Unique values in Category: ['Patisserie' 'Milk Products' 'Butchers' 'Beverages' 'Food' 'Furniture'
 'Electric household essentials' 'Computers and electric accessories']
Unique values in Payment Method: ['Digital Wallet' 'Credit Card' 'Cash']
Unique values in Location: ['Online' 'In-store']


In [94]:
# Identify major outliers in our numeric columns
numeric_cols = ['Price Per Unit', 'Quantity', 'Total Spent']

outlier_counts = {}

for col in numeric_cols:
    Q1 = sales_df[col].quantile(0.25)
    Q3 = sales_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR

    outliers = (sales_df[col] < lower_bound) | (sales_df[col] > upper_bound)

    outlier_counts[col] = outliers.sum()

outlier_counts = pd.DataFrame.from_dict(
    outlier_counts,
    orient='index',
    columns=['Extreme Outlier Count']
)

outlier_counts

,Extreme Outlier Count
Price Per Unit,0
Quantity,0
Total Spent,0


So, the two problems we've identified are the missing values and the dates being in the wrong format. Let's fix the dates first!

In [95]:
# Convert Transaction Date values to datetime
sales_df['Transaction Date'] = pd.to_datetime(sales_df['Transaction Date'], format='%Y-%m-%d')

sales_df['Transaction Date'].head()

,Transaction Date
0,2024-04-08
1,2023-07-23
2,2022-10-05
3,2022-05-07
4,2022-10-02


To handle null values, we can either drop rows with null values or impute data. Imputing means that we use logical techniques to fill in the missing values.

In [96]:
# Dropping all rows with nulls
sales_df_dropped = sales_df.dropna()
sales_df_dropped.shape

(7579, 11)

The problem with dropping all rows with nulls is that we've essentially gotten rid of half of our data.

In [97]:
# Impute 'Unknown' for missing discount
sales_df['Discount Applied'] = sales_df['Discount Applied'].fillna('Unknown')

In [98]:
# Show rows where Item is null
sales_df[sales_df['Item'].isnull()]

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
5,TXN_7482416,CUST_09,Patisserie,NaN,NaN,10.0,200.0,Credit Card,Online,2023-11-30,Unknown
7,TXN_1372952,CUST_21,Furniture,NaN,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,True
11,TXN_5422631,CUST_09,Milk Products,NaN,NaN,8.0,52.0,Digital Wallet,In-store,2025-01-12,True
15,TXN_1809665,CUST_14,Beverages,NaN,24.5,NaN,NaN,Credit Card,In-store,2022-05-11,Unknown
17,TXN_9634894,CUST_15,Milk Products,NaN,NaN,10.0,275.0,Digital Wallet,Online,2022-04-17,Unknown
...,...,...,...,...,...,...,...,...,...,...,...
12527,TXN_1069238,CUST_23,Food,NaN,5.0,NaN,NaN,Digital Wallet,In-store,2022-08-13,False
12552,TXN_4823896,CUST_05,Milk Products,NaN,8.0,NaN,NaN,Cash,In-store,2022-07-21,False
12556,TXN_4397672,CUST_04,Beverages,NaN,41.0,NaN,NaN,Credit Card,Online,2024-11-28,True
12562,TXN_7422454,CUST_07,Butchers,NaN,33.5,NaN,NaN,Cash,Online,2023-04-15,Unknown


In [104]:
# Drop rows with 2 or more nulls
sales_df_cleaned = sales_df.dropna(thresh=10)
sales_df_cleaned.shape

(11362, 11)

In [105]:
# Show null counts
null_counts = sales_df_cleaned.isnull().sum()
print(null_counts)

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


In [106]:
# Show data types of each column
sales_df_cleaned.dtypes

,0
Transaction ID,object
Customer ID,object
Category,object
Item,object
Price Per Unit,float64
Quantity,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


In [111]:
# Show summary statistics for the numerical data
sales_df_cleaned.describe(include="number")

,Price Per Unit,Quantity,Total Spent
count,11362.000000,11362.000000,11362.000000
mean,23.356847,5.535029,129.642537
std,10.736030,2.855166,94.662368
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,52.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [112]:
# Export cleaned dataset
sales_df_cleaned.to_csv('cleaned_retail_store_sales.csv', index=False)